# Train / Test Split for Pesticide–Respiratory Model

Loads the joint county-level dataset, drops rows missing targets (CASTHMA, COPD), and splits into **train** and **test** sets for modeling.

**Outputs:** `data/train.csv`, `data/test.csv`

County-level pesticide and health data have both **temporal** and **spatial** dimensions. The split strategy should match the type of generalisation you care about. We support **random**, **spatial (by state)**, and **spatio-temporal** (hold out states + last year(s)) splits.

**Note:** We are not making causal claims, and with only a few years (2016–2019) we do not have a rich time series—the temporal hold-out is mainly to avoid overfitting to particular years and places, not to support time-series modeling.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

DATA_DIR = "data"
JOINT_PATH = os.path.join(DATA_DIR, "joint_county_year_2016_2019.csv")
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

RANDOM_STATE = 42
TEST_SIZE = 0.2

## Splitting strategy: what kind of generalisation?

| Approach | What it tests | When to use |
|----------|----------------|--------------|
| **1. Simple random split** (with optional stratification) | Overall predictive accuracy when train and test are from the same population. Stratification keeps similar distributions of outcome, region, income, or cropland. | Data are roughly IID; you care about aggregate accuracy. Easy to implement. |
| **2. Time-based split** (temporal hold-out) | Generalisation to **future** years: train on earlier years, test on later. Avoids “peeking into the future”; mimics deployment. | You have **county–year** rows (panel data). *The joint dataset is county–year (joint_county_year_2016_2019.csv); you can train on 2016–2018 and test on 2019, or similar.* |
| **3. Spatial split** (state or region hold-out) | Generalisation to **unseen geography**: train on some states, test on others. Answers “does the model transfer to new states with different crop mixes and demographics?” | You care about transferring to new states or regions; use **split mode: spatial (by state)** below. |
| **4. Spatio-temporal split** | Hold out some states for test and restrict test to the **last year(s)** only. Train on other states × all years; test on held-out states × last 1–2 years. Reduces overfitting to place and time (we have limited years; this is not a rich time-series setup). | Use **split mode: spatio_temporal** below with county–year data. |

**Workflow:** Use **K-fold cross-validation on the training set** to tune hyperparameters and assess stability (avoids evaluating on the same data used for training). After choosing the best model, evaluate once on the **hold-out test set** (e.g. R², RMSE). We are not making causal claims; with only 2016–2019, temporal hold-out is to avoid overfitting to specific years/places.

## Load and filter

Keep only rows with non-missing targets (CASTHMA, COPD) so we can train and evaluate the model.

In [ ]:
df = pd.read_csv(JOINT_PATH)
print("Loaded:", df.shape[0], "rows,", df.shape[1], "columns")

targets = ["CASTHMA", "COPD"]
df_clean = df.dropna(subset=targets)
print("After dropping rows missing CASTHMA/COPD:", df_clean.shape[0], "rows")
print("Dropped", df.shape[0] - df_clean.shape[0], "rows")

## Split train / test

Choose **SPLIT_MODE** and (for random split) **STRATIFY_BY**.

- **`SPLIT_MODE = "random"`** — Random 80/20 split with optional stratification (below). Same population in train and test; good for overall accuracy.
- **`SPLIT_MODE = "spatial_state"`** — Hold out a set of **whole states** for test (~20% of states). Train on the rest. Tests spatial generalisation to unseen geography.
- **`SPLIT_MODE = "spatio_temporal"`** — Hold out ~20% of states and restrict **test to the last year(s)** only (e.g. 2019, or 2018–2019). Train = (all other states × all years). Test = (held-out states × those years). Requires county–year data with a `YEAR` column. With only a few years available, this guards against overfitting to time and place rather than supporting rich time-series inference.

**Stratification** (only when `SPLIT_MODE == "random"`):

| STRATIFY_BY | Why |
|-------------|-----|
| **outcome** (CASTHMA quartile) | Similar outcome distribution in train and test; fair evaluation. |
| **state** | Each state appears in both sets (no state-only train/test); small states grouped. |
| **income** | Balances low/medium/high income counties. |
| **cropland** | Balances agricultural vs. non-agricultural counties. |
| **None** | No stratification. |

In [ ]:
# "random" | "spatial_state" | "spatio_temporal"
SPLIT_MODE = "spatio_temporal"  # recommended: hold out states + last year(s); not causal, limited years
STRATIFY_BY = "outcome"  # only when SPLIT_MODE == "random"
TEST_LAST_YEARS = 1      # only when SPLIT_MODE == "spatio_temporal": test set = held-out states × last N years (e.g. 1 = 2019 only)

def _state_fips_series(df):
    return df["state_fips"].astype(int, errors="ignore").astype(str).str.zfill(2).replace("nan", "")

if SPLIT_MODE == "spatio_temporal":
    # Hold out some states; test = those states × last TEST_LAST_YEARS only. Train = other states × all years.
    import numpy as np
    if "YEAR" not in df_clean.columns:
        raise ValueError("SPLIT_MODE='spatio_temporal' requires county-year data with a YEAR column.")
    state_s = _state_fips_series(df_clean)
    states = state_s.dropna().unique()
    states = states[states != ""]
    n_states = len(states)
    n_holdout = max(1, int(round(n_states * TEST_SIZE)))
    rng = np.random.default_rng(RANDOM_STATE)
    test_states = set(rng.choice(states, size=n_holdout, replace=False))
    years = sorted(df_clean["YEAR"].dropna().unique())
    test_years = years[-TEST_LAST_YEARS:] if TEST_LAST_YEARS else years
    train_mask = (~state_s.isin(test_states))  # all years for non–held-out states
    test_mask = state_s.isin(test_states) & df_clean["YEAR"].isin(test_years)
    train_df = df_clean.loc[train_mask].copy()
    test_df = df_clean.loc[test_mask].copy()
    print("Split mode: spatio-temporal (hold out states + last year(s))")
    print("Held-out states (test):", sorted(test_states))
    print("Test years:", test_years)
    print("Train:", train_df.shape[0], "rows")
    print("Test:", test_df.shape[0], "rows")
elif SPLIT_MODE == "spatial_state":
    # Hold out a fraction of states for test; train on the rest. Reproducible via RANDOM_STATE.
    import numpy as np
    state_s = _state_fips_series(df_clean)
    states = state_s.dropna().unique()
    states = states[states != ""]
    n_states = len(states)
    n_holdout = max(1, int(round(n_states * TEST_SIZE)))
    rng = np.random.default_rng(RANDOM_STATE)
    test_states = set(rng.choice(states, size=n_holdout, replace=False))
    train_mask = state_s.isin(set(states) - test_states)
    test_mask = state_s.isin(test_states)
    train_df = df_clean.loc[train_mask].copy()
    test_df = df_clean.loc[test_mask].copy()
    print("Split mode: spatial (by state)")
    print("Held-out states (test):", sorted(test_states))
    print("Train:", train_df.shape[0], "rows ({:.0f}% of counties)".format(100 * train_df.shape[0] / len(df_clean)))
    print("Test:", test_df.shape[0], "rows ({:.0f}% of counties)".format(100 * test_df.shape[0] / len(df_clean)))
else:
    # Random split with optional stratification
    def make_stratify_column(df, by):
        if by is None or by == "none":
            return None
        if by == "outcome":
            return pd.qcut(df["CASTHMA"], q=4, labels=False, duplicates="drop")
        if by == "state":
            state_fips = _state_fips_series(df)
            state_counts = state_fips.value_counts()
            small_states = set(state_counts[state_counts < 10].index)
            return state_fips.map(lambda x: x if x not in small_states else "small")
        if by == "income":
            inc = df["median_income"].fillna(df["median_income"].median()).fillna(0)
            return pd.qcut(inc, q=4, labels=False, duplicates="drop")
        if by == "cropland":
            pc = df["pct_cropland"].fillna(0)
            return pd.qcut(pc.rank(method="first"), q=3, labels=False, duplicates="drop")
        return None

    strat_col = make_stratify_column(df_clean, STRATIFY_BY)
    stratify_kw = {"stratify": strat_col} if strat_col is not None else {}
    train_df, test_df = train_test_split(
        df_clean, test_size=TEST_SIZE, random_state=RANDOM_STATE, **stratify_kw
    )
    for c in ["casthma_quartile", "stratify_temp"]:
        if c in train_df.columns:
            train_df = train_df.drop(columns=[c])
        if c in test_df.columns:
            test_df = test_df.drop(columns=[c])
    print("Split mode: random (stratify by:", STRATIFY_BY or "None", ")")
    print("Train:", train_df.shape[0], "rows")
    print("Test:", test_df.shape[0], "rows")

## Save

**Next steps:** Use **K-fold cross-validation on the training set** to tune hyperparameters and assess stability (avoids evaluating on the same data used for training). After selecting the best model, evaluate it **once** on this hold-out test set (e.g. R², RMSE).

In [ ]:
train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
print("Saved:", TRAIN_PATH, "|", TEST_PATH)